# Lattice reactivity surrogate — interactive analysis

Loads the campaign store and works through the same analysis as
`scripts/train_surrogate.py`, one step at a time, so each claim can be
poked at rather than read off a report.

Run the campaign first:

```bash
docker compose -f docker/docker-compose.yml run --rm campaign
```

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from lattice_uq import BOUNDS, PARAM_NAMES
from lattice_uq.store import ResultStore

store = ResultStore(ROOT / "data")
df = store.to_frame()
print(f"{len(df)} runs")
df.groupby("tag").agg(
    n=("keff", "size"),
    k_min=("keff", "min"),
    k_max=("keff", "max"),
    mean_sigma_pcm=("keff_sigma_pcm", "mean"),
    mean_wall_s=("wall_time_s", "mean"),
)

## 1. Is the reported σ believable?

Before comparing anything *to* σ, check σ itself. The replicate runs repeat a
design point under different RNG seeds, so the sample scatter across seeds can
be compared directly with the σ OpenMC reported. A ratio well above 1 would
mean the quoted uncertainty understates the real spread — the classic symptom
of inter-batch correlation from an under-converged fission source — and would
undermine every later claim.

In [ ]:
reps = df[df.tag == "replicate"]
summary = reps.groupby("run_id").agg(
    n_seeds=("keff", "size"),
    mean_k=("keff", "mean"),
    observed_std_pcm=("keff", lambda s: s.std(ddof=1) * 1e5),
    reported_sigma_pcm=("keff_sigma_pcm", "mean"),
)
summary["ratio"] = summary.observed_std_pcm / summary.reported_sigma_pcm
print(f"mean observed/reported = {summary.ratio.mean():.2f}")
summary

## 2. The response surface

One-dimensional marginals of a 4-D LHS: each panel is every run plotted
against one parameter, so the trends are visible through the scatter caused by
the other three. Useful as a sanity check before fitting anything — the
enrichment and moderator-density trends should be obvious by eye, the Doppler
trend weak but negative.

In [ ]:
import matplotlib.pyplot as plt

from lattice_uq import plotting

plotting.apply_style()
train = df[df.tag == "train"]
fig, axes = plt.subplots(1, len(PARAM_NAMES), figsize=(3.1 * len(PARAM_NAMES), 3.2),
                         sharey=True)
for ax, name in zip(axes, PARAM_NAMES):
    ax.scatter(train[name], train.keff, s=14, color=plotting.BLUE,
               edgecolor=plotting.SURFACE, linewidth=0.4)
    ax.set_xlabel(name.replace("_", " "))
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
axes[0].set_ylabel("k-infinity")
fig.suptitle("k-infinity marginals", x=0.005, ha="left", fontweight="semibold")
fig.tight_layout()

## 3. Fit the Gaussian process

The per-point Monte Carlo variance is passed in as the GP's `alpha`, so the
model is solving the correct heteroscedastic problem rather than assuming every
run is equally sharp.

In [ ]:
from lattice_uq.surrogate import (
    GPSurrogate,
    design_matrix,
    evaluate,
    uncertainty_comparison,
)

test = df[df.tag == "test"]
x_tr, y_tr, s_tr = design_matrix(train), train.keff.to_numpy(), train.keff_sigma.to_numpy()
x_te, y_te, s_te = design_matrix(test), test.keff.to_numpy(), test.keff_sigma.to_numpy()

gp = GPSurrogate().fit(x_tr, y_tr, s_tr)
pred, pred_std = gp.predict(x_te, return_std=True)
metrics = evaluate(y_te, pred, pred_std, s_te)
metrics

In [ ]:
# Fitted length scales on the unit cube: a short length scale means k varies
# quickly in that direction, a very long one means the response is nearly flat
# in it. This is the GP telling you which parameters actually matter.
gp.kernel_summary["length_scales_unit_cube"]

## 4. The headline comparison

Held-out surrogate error against the stochastic uncertainty of the transport
runs themselves.

In [ ]:
comparison = uncertainty_comparison(metrics, s_te, s_tr)
for k, v in comparison.items():
    print(f"{k:<28} {v}")

In [ ]:
abs_err_pcm = np.abs(pred - y_te) * 1e5
plotting.plot_error_vs_sigma(
    abs_err_pcm, s_te * 1e5, ROOT / "figures/nb_error_vs_sigma.png",
    gp_sigma_pcm=pred_std * 1e5,
)

## 5. Reactivity coefficients

Derivatives of the *fitted surface*, with uncertainty propagated from the GP's
joint posterior at the two stencil points. The fuel temperature coefficient
must be negative — Doppler broadening of the ²³⁸U capture resonances increases
absorption as the fuel heats. The surrogate was never told that.

In [ ]:
from lattice_uq.surrogate import EXPECTED_SIGNS, coefficient

centre = np.array([0.5 * (BOUNDS[n].low + BOUNDS[n].high) for n in PARAM_NAMES])
rows = []
for param in EXPECTED_SIGNS:
    c = coefficient(gp, centre, param)
    rows.append({
        "parameter": param,
        "value": c.value_pcm_per_unit,
        "sigma": c.sigma_pcm_per_unit,
        "units": c.units,
        "expected": "negative" if c.expected_sign < 0 else "positive",
        "consistent": c.consistent,
    })
pd.DataFrame(rows)

In [ ]:
temps = np.linspace(630, 1170, 25)
vals, sigs = [], []
for t in temps:
    p = centre.copy()
    p[PARAM_NAMES.index("fuel_temperature")] = t
    c = coefficient(gp, p, "fuel_temperature")
    vals.append(c.value_pcm_per_unit)
    sigs.append(c.sigma_pcm_per_unit)

plotting.plot_coefficient(
    temps, vals, sigs, ROOT / "figures/nb_doppler.png",
    xlabel="Fuel temperature (K)",
    title="Doppler coefficient from the surrogate",
    note="Must be negative everywhere. A PWR value is of order -2 to -4 pcm/K.",
)